# AmazonQA-RAG — Phase 1: QLoRA Fine-tuning of Mistral-7B on Amazon Customer QA

> **Project:** AmazonQA-RAG — Fine-tuned LLM + Retrieval Augmented Generation with RAGAS Evaluation  
> **Author:** [Your Name] · BTech CS (Data Science), Final Year  
> **Target venue:** Amazon ML Summer School 2026 application portfolio

---

## What this notebook does

This is **Phase 1 of 4** in the AmazonQA-RAG pipeline. We fine-tune a 7B-parameter instruction-tuned LLM on real Amazon customer questions using **QLoRA** — 4-bit quantization of the base weights plus trainable Low-Rank Adaptation matrices. The full 7B model would need ~28 GB of VRAM in fp16; we'll squeeze it into a free Colab T4 (15 GB) and still get a faithful fine-tune.

### Pipeline overview

```
    Amazon QA (HF)              Mistral-7B-Instruct-v0.2
          │                                │
          ▼                                ▼
  [ flatten + format ]          [ BitsAndBytes NF4 4-bit ]
          │                                │
          ▼                                ▼
  ┌────────────────────────────────────────────────────┐
  │  SFTTrainer + LoRA(r=16, α=32) on q/k/v/o/MLP      │
  │  optim=paged_adamw_8bit, lr=2e-4, cosine schedule  │
  └────────────────────────────────────────────────────┘
          │
          ▼
  artifacts/lora_adapter/   ←  ~80 MB adapter, ready for RAG (Phase 2)
```

### Why QLoRA, not full fine-tuning?

| Approach | VRAM (7B) | Trainable params | Time on T4 |
|---|---|---|---|
| Full FT (fp16) | ~28 GB | 7,000,000,000 | impossible |
| LoRA (fp16 base) | ~16 GB | ~40 M | OOM on T4 |
| **QLoRA (4-bit base)** | **~10 GB** | **~40 M** | **~45–60 min** |

QLoRA (Dettmers et al., 2023) is the only path that fits on free-tier hardware while preserving 99%+ of full-fine-tune quality on instruction-following tasks.

### Hardware check

We need a CUDA GPU. On Colab: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# Sanity check: GPU + VRAM
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

---
## 1. Install dependencies

Pinned versions for reproducibility. The `bitsandbytes` + `transformers` + `peft` + `trl` quartet is famously brittle across versions — these specific versions are known to work together on T4 (CUDA 12.1) as of mid-2025.

In [ ]:
%%capture
# Pin NumPy 1.x FIRST — Colab's NumPy 2.x is binary-incompatible with the
# older transformers/datasets/bitsandbytes extensions we need.
!pip install -q "numpy<2"

!pip install -q -U \
    transformers==4.44.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.3 \
    accelerate==0.33.0 \
    datasets==2.21.0 \
    sentencepiece==0.2.0 \
    huggingface_hub==0.24.6

# Re-pin NumPy in case any of the above re-pulled 2.x as a dependency.
!pip install -q --force-reinstall "numpy<2"

### RESTART THE RUNTIME (required after the NumPy downgrade)

Python has the old NumPy 2.x already loaded — we must kill the kernel so the new NumPy 1.x is picked up. The cell below kills the Python process; Colab will show **"Your session crashed"** which is **intentional and expected**.

After the auto-restart, **skip the install cell and this kill cell** (they've done their job) and continue from the HuggingFace login cell below.

In [ ]:
# Kill the kernel — Colab restarts automatically. 'Session crashed' is expected.
import os
os.kill(os.getpid(), 9)

### Authenticate with HuggingFace (required for Mistral)

Mistral-7B-Instruct-v0.2 is a gated model — you must accept its license once on the HF model page and then provide a read-token here. Get yours at: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
import os

# Option A — interactive (recommended in Colab):
# login()

# Option B — Colab secrets (safer, no token in the notebook):
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in to HuggingFace via Colab secret.')
except Exception as e:
    print('No Colab secret found. Falling back to interactive login.')
    print('If you skip this, you can still run the Phi-2 fallback (no auth required).')
    # login()

---
## 2. Clone the project repo (or upload `configs/` + `src/`)

This notebook depends on the project's `configs/config.py` and `src/finetune.py` modules. In a real submission you'd `git clone` your GitHub repo here:

In [ ]:
# Replace with your own repo URL once you push to GitHub:
# !git clone https://github.com/<your-username>/AmazonQA-RAG.git
# %cd AmazonQA-RAG

# For local-only runs, set PROJECT_ROOT manually:
import sys, os
PROJECT_ROOT = '/content/AmazonQA-RAG'
os.makedirs(PROJECT_ROOT, exist_ok=True)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Make sure configs/config.py and src/finetune.py are present under it.')

---
## 3. Load configuration

All hyperparameters live in `configs/config.py` as dataclasses. The notebook never hardcodes a value. To switch from Mistral-7B to Phi-2 (the fallback for tighter VRAM budgets), use `get_phi2_fallback_config()` instead — one line.

In [ ]:
from configs.config import get_default_config, get_phi2_fallback_config

# Choose your base model here:
USE_PHI2_FALLBACK = False  # set True if Mistral OOMs on your T4

cfg = get_phi2_fallback_config() if USE_PHI2_FALLBACK else get_default_config()

print(f'Base model         : {cfg.model.base_model_id}')
print(f'LoRA targets       : {cfg.lora.target_modules}')
print(f'Train sample size  : {cfg.dataset.train_sample_size:,}')
print(f'Max seq length     : {cfg.dataset.max_seq_length}')
print(f'Effective batch    : {cfg.training.per_device_train_batch_size * cfg.training.gradient_accumulation_steps}')
print(f'Adapter output dir : {cfg.paths.adapter_dir}')

---
## 4. Load & preprocess the Amazon QA dataset

We use `embedding-data/Amazon-QA` from the HuggingFace Hub — ~190 k real customer Q→A pairs scraped from Amazon product pages. Each raw row looks like:

```python
{"query": "Is this charger compatible with iPhone 13?",
 "pos":   ["Yes, it's a standard USB-C ...", "Should work fine ..."]}
```

Our preprocessing:
1. **Sub-sample** to `train_sample_size` (10k) so the run fits in ~1 hour on T4.
2. **Flatten** `{query, pos:[...]}` → `{question, answer}` using the top-1 answer.
3. **Filter** out junk rows (Q or A shorter than 5 chars).
4. **Format** with Mistral's `[INST] … [/INST]` template (defined in config).
5. **Split** 95/5 train/validation.
6. **Save** to disk for reproducibility — Phase 3 (evaluation) will reuse the same split.

In [ ]:
from src.finetune import load_and_preprocess_dataset

dataset = load_and_preprocess_dataset(cfg)
dataset

In [ ]:
# Inspect one formatted training example
print(dataset['train'][0]['text'])
print('\n--- length stats ---')
import numpy as np
lens = [len(x['text']) for x in dataset['train'].select(range(min(1000, len(dataset['train']))))]
print(f'min={min(lens)}  median={int(np.median(lens))}  max={max(lens)}  mean={int(np.mean(lens))}')

---
## 5. Load the base LLM in 4-bit (QLoRA recipe)

We use **BitsAndBytes NF4** — the 4-bit data type from the QLoRA paper, designed so that the resulting quantized weights are information-theoretically optimal for normally-distributed neural net weights. Combined with **double quantization** (quantizing the quantization constants themselves), we shave another ~0.4 bits/parameter.

Compute dtype is **bfloat16** — gradients and intermediate activations stay in bf16, only the frozen weights are 4-bit.

In [ ]:
from src.finetune import load_base_model_and_tokenizer

model, tokenizer = load_base_model_and_tokenizer(cfg)
print(model)

In [ ]:
# How much VRAM did the 4-bit base actually take?
import torch
alloc_gb = torch.cuda.memory_allocated() / 1024**3
reserv_gb = torch.cuda.memory_reserved() / 1024**3
print(f'Allocated : {alloc_gb:.2f} GB')
print(f'Reserved  : {reserv_gb:.2f} GB')
print(f'(Mistral-7B in 4-bit + double-quant should be ~5 GB)')

---
## 6. Smoke-test BEFORE fine-tuning (baseline)

Let's see how the off-the-shelf instruction-tuned model answers an Amazon-style question. We'll re-run the exact same prompt after training to measure subjective improvement.

In [ ]:
from src.finetune import generate_with_model

SAMPLE_QUESTION = (
    'Does the Echo Dot 5th gen work with Apple Music, and can I use it as a Bluetooth '
    'speaker for my iPhone at the same time?'
)

prompt = cfg.prompt_template.format(question=SAMPLE_QUESTION, answer='').rstrip()

print('=== BEFORE fine-tuning ===\n')
baseline_answer = generate_with_model(model, tokenizer, prompt, max_new_tokens=180)
print(baseline_answer)

---
## 7. Attach LoRA adapters

We freeze the 4-bit base and inject trainable low-rank matrices `ΔW = B·A` (where `A ∈ ℝ^{r×d}`, `B ∈ ℝ^{d×r}`, `r=16`) into the attention and MLP projections. Only ~40 M of the model's ~7 B params become trainable — that's the magic of LoRA.

In [ ]:
from src.finetune import attach_lora_adapters

model = attach_lora_adapters(model, cfg)
model.print_trainable_parameters()

---
## 8. Train with SFTTrainer

`SFTTrainer` from TRL is a thin wrapper around HuggingFace `Trainer` specialised for instruction-style supervised fine-tuning. Key choices:

- **`paged_adamw_8bit`** — bitsandbytes' paged optimizer; offloads optimizer states to CPU on demand, halving peak VRAM.
- **`gradient_checkpointing=True`** — recompute activations instead of storing them. ~30 % slower but cuts activation memory ~3×.
- **`group_by_length=True`** — bucket samples by token count so each batch has minimal padding. ~20 % throughput win on variable-length QA data.
- **`packing=False`** — packing concatenates multiple short examples per sequence for higher token throughput, but on QA-style instruction data it noticeably hurts answer-faithfulness, so we leave it off.

In [ ]:
from src.finetune import build_trainer

trainer = build_trainer(model, tokenizer, dataset, cfg)

In [ ]:
# This is the long-running cell — expect ~45–60 min on a T4 with the default 10k samples.
# If you need a quick demo run, set cfg.training.max_steps = 100 BEFORE this cell.
trainer.train()

---
## 9. Save the LoRA adapter

We save *only* the LoRA adapter weights (~80 MB) — never the full 14 GB base model. Phase 2 and beyond will reload the base from HF and reapply this adapter.

In [ ]:
trainer.model.save_pretrained(str(cfg.paths.adapter_dir))
tokenizer.save_pretrained(str(cfg.paths.adapter_dir))

# Verify
!ls -lh {cfg.paths.adapter_dir}

### Persist to Google Drive (optional but recommended)

Colab sessions die after ~12 h of idle time and the local disk evaporates with them. Copy the adapter to Drive so Phases 2–4 can resume from a clean session.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DEST = '/content/drive/MyDrive/AmazonQA-RAG/lora_adapter'
    !mkdir -p "{DRIVE_DEST}"
    !cp -r "{cfg.paths.adapter_dir}"/* "{DRIVE_DEST}/"
    print(f'\nAdapter persisted to: {DRIVE_DEST}')
except Exception as e:
    print(f'(Skipping Drive backup — not on Colab or user declined: {e})')

---
## 10. Smoke-test AFTER fine-tuning

Re-run the same baseline prompt and look for: better adherence to Amazon-style conversational tone, fewer hallucinated product features, more decisive yes/no language typical of seller-Q&A responses.

In [ ]:
print('=== AFTER fine-tuning ===\n')
ft_answer = generate_with_model(trainer.model, tokenizer, prompt, max_new_tokens=180)
print(ft_answer)

print('\n\n=== Side-by-side ===')
print('BEFORE:', baseline_answer[:300])
print('AFTER :', ft_answer[:300])

---
## 11. Free GPU memory

Run this before launching Phase 2 in the same session — otherwise you'll OOM when stacking the embedding model on top of the still-resident LLM.

In [ ]:
import gc, torch

del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated')

---
## Phase 1 complete

**Deliverable:** `artifacts/lora_adapter/` — an ~80 MB LoRA adapter that, when stacked on top of 4-bit Mistral-7B-Instruct-v0.2, produces a model specialized on Amazon customer-Q&A patterns.

### Phase 1 → Phase 2 handoff

Phase 2 (RAG pipeline) will:
1. Build a FAISS index over an Amazon product/support corpus.
2. Wrap the fine-tuned model behind a LangChain `RetrievalQA` chain.
3. Compare retrieval-augmented vs. parametric-only answers on the same prompts you tried above.

### Troubleshooting cheat-sheet

| Symptom | Fix |
|---|---|
| `CUDA out of memory` during model load | Switch to Phi-2 (`USE_PHI2_FALLBACK = True`) |
| `CUDA OOM` mid-training | Reduce `per_device_train_batch_size` to 2 and double `gradient_accumulation_steps` |
| `OSError: Mistral gated` | Accept the license on the HF model page, then re-login |
| `bitsandbytes` import error | Restart runtime — version mismatch with torch |
| Loss stuck at ~2.0+ | Verify the prompt template matches `cfg.prompt_template` |